# Anexo A3. Evaluación avanzada

**Opcional** · **Dificultad: alta** · Complementa al notebook 3

1. nDCG@k (ejercicio)
2. Calibración del judge con etiquetas humanas: acuerdo y kappa de Cohen
3. Contraste con una librería: ranx

In [1]:
from rag_lab import CFG, set_seeds, summary

# CFG.FAST_MODE = False  # corpus de control completo (por defecto se lee FAST_MODE de .env)
# CFG.DOC_PATH = CFG.DATA_DIR / "mi_documento.pdf"  # cualquier PDF sirve
set_seeds()
summary()

Documento:   /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/data/documento.pdf (existe)
FAST_MODE:   True
Dispositivo: cuda
GEN_MODEL:   google/gemini-3.1-flash-lite
JUDGE_MODEL: openai/gpt-5-mini
VLM_MODEL:   google/gemini-3.1-flash-lite
OpenRouter:  clave configurada


Trazas:      jsonl /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/trazas/trazas.jsonl  (alumno: profesor)


/home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/rag_lab/tracing.py:49: UserWarning: No se pudo conectar con Langfuse ([Errno 111] Connection refused): las trazas se guardan en JSONL.
  warnings.warn(f"No se pudo conectar con Langfuse ({e}): las trazas se guardan en JSONL.")


In [2]:
import pandas as pd
from rag_lab import evaluation

questions = evaluation.load_questions()
df_questions = pd.DataFrame(questions)
print(df_questions["tipo"].value_counts().to_dict(), "| pendientes de revisar:", int(df_questions["revisar_manual"].sum()))
df_questions[["id", "tipo", "paginas", "tabla_gemela", "pregunta"]]

{'texto': 7, 'tabla': 7, 'visual': 6} | pendientes de revisar: 20


,id,tipo,paginas,tabla_gemela,pregunta
0,q01,texto,[2],NaN,¿Qué línea se centra en la promoción del equil...
1,q02,texto,[3],NaN,Cuál es la duración total del proyecto según e...
2,q03,texto,[6],NaN,¿Cuál es el nombre del profesor del Centro de ...
3,q04,texto,[11],NaN,¿Cuántas fases principales componen el desarro...
4,q05,texto,[9],NaN,¿Qué centro de innovación y desarrollo para la...
5,q06,texto,[7],NaN,¿Cuál es la titulación del desarrollador backe...
6,q07,texto,[8],NaN,¿Cuál es el objetivo de la Tarea 9 dentro de l...
7,q08,tabla,[18],17.0,¿Qué instrumento se utiliza para medir el impa...
8,q09,tabla,[17],17.0,¿Qué instrumento se utiliza para medir el nive...
9,q10,tabla,[14],13.0,¿Cuántas empresas se beneficiarán por curso se...


In [3]:
def recall_k(retrieved, correct, k):
    return float(any(set(u) & set(correct) for u in retrieved[:k]))

def mrr_k(retrieved, correct, k):
    for position, unit in enumerate(retrieved[:k], start=1):
        if set(unit) & set(correct):
            return 1 / position
    return 0.0

example = [[3], [7, 8], [12], [13]]  # páginas de los fragmentos recuperados, en orden
print(recall_k(example, [12], 2), recall_k(example, [12], 5), mrr_k(example, [12], 10), mrr_k(example, [8], 10))
assert mrr_k(example, [8], 10) == evaluation.mrr_at_k(example, [8], 10)

0.0 1.0 0.3333333333333333 0.5


Evaluamos V3 (híbrido) pregunta a pregunta, con el judge.

In [4]:
from rag_lab import versions

v3 = versions.build("V3")
eval_v3 = evaluation.evaluate_rag(v3, questions, with_judge=True, register=False)
detail = eval_v3["detalle"]
detail["posición correcta"] = [next((i for i, u in enumerate(r, 1) if set(u) & set(c)), None)
                                for r, c in zip(detail["recuperadas"], detail["correctas"])]
detail["recall@5"] = [recall_k(r, c, 5) for r, c in zip(detail["recuperadas"], detail["correctas"])]
detail["MRR@10"] = [mrr_k(r, c, 10) for r, c in zip(detail["recuperadas"], detail["correctas"])]
detail[["id", "tipo", "correctas", "posición correcta", "recall@5", "MRR@10"]]

,id,tipo,correctas,posición correcta,recall@5,MRR@10
0,q01,texto,[2],1,1.0,1.000000
1,q02,texto,[3],2,1.0,0.500000
2,q03,texto,[6],1,1.0,1.000000
3,q04,texto,[11],1,1.0,1.000000
4,q05,texto,[9],1,1.0,1.000000
5,q06,texto,[7],1,1.0,1.000000
6,q07,texto,[8],1,1.0,1.000000
7,q08,tabla,[18],1,1.0,1.000000
8,q09,tabla,[17],1,1.0,1.000000
9,q10,tabla,[14],1,1.0,1.000000


## 1. nDCG@k

MRR solo mira la primera unidad correcta. nDCG (*normalized Discounted Cumulative Gain*) suma todas las correctas
con un descuento logarítmico por posición y lo divide entre el valor ideal.

In [5]:
# TODO: implementa nDCG@k con relevancia binaria por página: DCG = suma de 1 / log2(posición + 1) para las
# unidades correctas; IDCG es el DCG ideal (una sola unidad correcta en la posición 1, es decir, 1).
# Cuenta como mucho un acierto por página correcta.
import math

def ndcg_k(retrieved, correct, k):
    # Completa
    return None

print([ndcg_k(r, c, 10) for r, c in zip(detail["recuperadas"][:5], detail["correctas"][:5])])

[None, None, None, None, None]


## 2. Calibración del judge con etiquetas humanas

Un judge solo es útil si coincide con lo que diría una persona. `data/etiquetas_humanas.csv` tiene 20 elementos
(pregunta, respuesta, contexto) de V0 y V3 con las columnas `*_humano` vacías para que el profesor ponga 1 o 0.
Se mide el porcentaje de acuerdo y la kappa de Cohen (acuerdo corregido por azar: 0 = como el azar, 1 = perfecto).
El fichero nunca se sobrescribe.

In [6]:
from rag_lab import judge

v0 = versions.build("V0")
eval_v0 = evaluation.evaluate_rag(v0, questions, with_judge=True, register=False)
items = []
for i, p in enumerate(questions):
    source, version = (eval_v0, "V0") if i % 2 == 0 else (eval_v3, "V3")
    row = source["detalle"].iloc[i]
    items.append({"id": p["id"], "version": version, "pregunta": p["pregunta"], "respuesta": row["respuesta"],
                  "contexto": row["contexto"], **{c: row[c] for c in judge.CRITERIA}})
created = judge.create_label_template(items)
print("Plantilla creada ahora" if created else "La plantilla ya existía: no se toca", "->", CFG.LABELS_PATH)

La plantilla ya existía: no se toca -> /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/data/etiquetas_humanas.csv


In [7]:
labels = judge.load_labels()
if labels is not None:
    judge_verdicts = pd.DataFrame(items).set_index("id")
    rows = []
    for criterion in judge.CRITERIA:
        human = labels[f"{criterion}_humano"].tolist()
        machine = judge_verdicts.loc[labels["id"], criterion].astype(int).tolist()
        rows.append({"criterio": criterion, **judge.agreement(machine, human)})
    display(pd.DataFrame(rows).set_index("criterio").round(3))

etiquetas_humanas.csv todavía no tiene etiquetas humanas (columnas *_humano vacías). Se continúa sin calibración: rellénalas con 1 (sí) o 0 (no) y vuelve a ejecutar esta celda.


## 3. Contraste con una librería: ranx

Comprobamos nuestras funciones en el corpus de control (SQAC) frente a `ranx`. Con un único documento relevante
por pregunta, nuestro recall@k coincide con `hit_rate@k` y `recall@k` de ranx, y MRR@k con `mrr@k`.
La primera vez ranx compila sus funciones (tarda unos segundos).

In [8]:
import warnings

warnings.filterwarnings("ignore")
from ranx import Qrels, Run, evaluate
from rag_lab.control_corpus import load_sqac
from rag_lab.indexes import BM25Index

sqac = load_sqac()
units = [{"id": i, "texto": t, "paginas": [i]} for i, t in sqac["docs"].items()]
bm25 = BM25Index(units)
rankings = {q["id"]: bm25.search(q["texto"], 10) for q in sqac["consultas"]}

qrels = Qrels({q["id"]: {q["doc"]: 1} for q in sqac["consultas"]})
run = Run({qid: {d: s for d, s in r} or {"vacio": 0.0} for qid, r in rankings.items()})
from_ranx = evaluate(qrels, run, ["hit_rate@5", "recall@10", "mrr@10"])
ours = evaluation.evaluate_retrieval(
    [{"recuperadas": [[d] for d, _ in rankings[q["id"]]], "correctas": [q["doc"]]} for q in sqac["consultas"]])
pd.DataFrame({"rag_lab": [ours["recall@5"], ours["recall@10"], ours["MRR@10"]],
              "ranx": [from_ranx["hit_rate@5"], from_ranx["recall@10"], from_ranx["mrr@10"]]},
             index=["recall@5 / hit_rate@5", "recall@10", "MRR@10"]).round(4)

,rag_lab,ranx
recall@5 / hit_rate@5,0.7750,0.7750
recall@10,0.8400,0.8400
MRR@10,0.6694,0.6694


In [9]:
from rag_lab import models

models.release()

## Qué hemos visto

- nDCG premia tener varias unidades correctas arriba; con una sola página correcta se parece a MRR.
- Un judge sin calibrar es una opinión: hay que medir su acuerdo (kappa) con etiquetas humanas.
- Nuestras métricas coinciden con las de una librería de evaluación (ranx).

## Ejercicio propuesto

Etiqueta tú mismo 10 de las 20 filas en una copia de `etiquetas_humanas.csv` y calcula la kappa de Cohen entre tus
etiquetas y las del judge para la fidelidad.